# AV1 - Analise e Visualizacao de Dados (2026.2)
## Grupo 6 - EcoVolt Analytics

**Notebook de curadoria, tratamento e visualizacao** alinhado a ficha do professor para o grupo EcoVolt:
> "Das cargas criticas a comparacao de cenarios. Como energia, potencia e demanda condicionam a autonomia."

Este notebook demonstra, em sequencia:
1. **Recorte e decisao** (perguntas P1-P3 e classificacao real/estimado/simulado);
2. **Inventario e dicionario** das fontes (com hash de rastreabilidade);
3. **Curadoria**: cinco verificacoes de qualidade, tratamento e registro das decisoes;
4. **Indicadores**: formulas, unidades, casos especiais (carga zero / potencia insuficiente);
5. **Visualizacoes V1, V2 e V3** justificadas;
6. **Conferencia independente** (exemplo do professor + conta paralela);
7. **Sintese textual e limitacoes**.

**Como rodar:** execute as celulas em ordem. O kernel precisa de `pandas`, `numpy`, `plotly` e `openpyxl`.
A celula de configuracao localiza a pasta do projeto automaticamente.


## 0. Recorte, decisao e perguntas

**Decisao apoiada:** *"Nas condicoes informadas, o cenario atende as cargas criticas durante o tempo requerido?"*

| ID | Pergunta analitica | Vista |
|----|--------------------|-------|
| P1 | Quais cargas formam a demanda critica? | V1 |
| P2 | Qual cenario atende ao horizonte requerido? | V2 |
| P3 | Como muda a autonomia quando a premissa (SOH/energia) varia? | V3 |

**Fronteiras:** operacao simulada (posto de saude, telecom, residencia). Unidade de analise = um cenario de banco + horizonte.
**Fica fora:** projeto eletrico real, ensaio de bateria, viabilidade comercial, economia.

**Classificacao das origens (exigencia do professor):**

| Elemento | Classificacao | Origem |
|---|---|---|
| Dataset SOH CALCE CS2 | observado | dados/raw (arquivo mal nomeado) |
| Curva de carga 5,596 Ah | observado | dados/raw |
| Cargas e duracao dos cenarios | simulado (declarado) | premissa do grupo |
| fator_capacidade, fracao_uso, eficiencia | estimado (premissa) | CALCE + premissa |


In [4]:
import hashlib
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


def encontrar_raiz(inicio=None):
    """Sobe a arvore de diretorios ate achar a pasta do projeto."""
    p = Path(inicio or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "dados").exists() and (cand / "analise").exists():
            return cand
    raise FileNotFoundError("Nao encontrei a pasta do projeto (dados/ + analise/).")


BASE = encontrar_raiz()
RAW = BASE / "dados" / "raw"
PROC = BASE / "dados" / "processados"
VISTAS = BASE / "dashboard" / "vistas"
PROC.mkdir(parents=True, exist_ok=True)
VISTAS.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", BASE)
print("Fontes em dados/raw:")
for f in sorted(RAW.iterdir()):
    print(f"  - {f.name}  ({f.stat().st_size/1024:.1f} KB)")


Raiz do projeto: C:\Users\pedro\OneDrive\Desktop\Análise
Fontes em dados/raw:
  - ORIGINAL_5.596Ah_charge.xlsx  (204.7 KB)
  - ORIGINAL_5_596Ah_charge.csv  (288.0 KB)
  - ORIGINAL_nomeErrado_CALCE_CS2_ML_SOH_Dataset.csv  (204.7 KB)
  - ORIGINAL_nomeErrado_recarregados-emplacamentos.csv  (503.0 KB)


## 1. Inventario e dicionario

Grão/chave das fontes usadas. O hash SHA-256 liga cada original tratado ao arquivo de origem (rastreabilidade).

In [5]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


inventario = pd.DataFrame([
    {"arquivo": "ORIGINAL_nomeErrado_recarregados-emplacamentos.csv",
     "conteudo_real": "Dataset SOH CALCE CS2", "grao": "linha = ciclo de uma celula",
     "chave": "battery_id + cycle_number", "classificacao": "observado"},
    {"arquivo": "ORIGINAL_5_596Ah_charge.csv",
     "conteudo_real": "Curva de carga 5,596 Ah", "grao": "linha = leitura de 1 s",
     "chave": "Date_Time", "classificacao": "observado"},
    {"arquivo": "ORIGINAL_5.596Ah_charge.xlsx",
     "conteudo_real": "Curva de carga (planilha)", "grao": "linha = leitura de 1 s",
     "chave": "Date_Time", "classificacao": "observado"},
    {"arquivo": "ORIGINAL_nomeErrado_CALCE_CS2_ML_SOH_Dataset.csv",
     "conteudo_real": "XLSX mal nomeado (igual a curva)", "grao": "-",
     "chave": "-", "classificacao": "observado"},
])
inventario["sha256"] = inventario["arquivo"].map(lambda n: sha256(RAW / n)[:16] + "...")
inventario


,arquivo,conteudo_real,grao,chave,classificacao,sha256
0,ORIGINAL_nomeErrado_recarregados-emplacamentos...,Dataset SOH CALCE CS2,linha = ciclo de uma celula,battery_id + cycle_number,observado,de823114bf77616f...
1,ORIGINAL_5_596Ah_charge.csv,"Curva de carga 5,596 Ah",linha = leitura de 1 s,Date_Time,observado,cfbe14877c26c33d...
2,ORIGINAL_5.596Ah_charge.xlsx,Curva de carga (planilha),linha = leitura de 1 s,Date_Time,observado,072261ea7a95afc6...
3,ORIGINAL_nomeErrado_CALCE_CS2_ML_SOH_Dataset.csv,XLSX mal nomeado (igual a curva),-,-,observado,072261ea7a95afc6...


## 2. Curadoria (limpeza e tratamento)

### 2.1 Achado de qualidade - arquivos com nome trocado
O arquivo `...CALCE_CS2_ML_SOH_Dataset.csv` **nao e CSV**: sua assinatura comeca com `PK` (assinatura de ZIP/XLSX) e ele e identico ao `5.596Ah_charge.xlsx`. O verdadeiro dataset SOH CALCE chegou com o nome `...recarregados-emplacamentos.csv` (que **nao** sao emplacamentos).

In [6]:
alvo = RAW / "ORIGINAL_nomeErrado_CALCE_CS2_ML_SOH_Dataset.csv"
assinatura = alvo.read_bytes()[:4]
print("Assinatura (4 bytes):", assinatura)
print("Diagnostico:", "XLSX disfarcado de .csv" if assinatura[:2] == b"PK" else "texto")
try:
    pd.read_csv(alvo, nrows=2)
    print("Leu como CSV.")
except Exception as e:
    print(f"Falha ao ler como CSV -> {type(e).__name__}")
    print("Confirmado: extensao .csv, conteudo binario de planilha.")


Assinatura (4 bytes): b'PK\x03\x04'
Diagnostico: XLSX disfarcado de .csv
Falha ao ler como CSV -> UnicodeDecodeError
Confirmado: extensao .csv, conteudo binario de planilha.


### 2.2 As cinco verificacoes de qualidade (CALCE SOH)

In [7]:
def verificar_qualidade(df, chave):
    return {
        "1. chaves/duplicidade":
            f"{int(df.duplicated().sum())} linhas duplicadas; "
            f"{int(df.duplicated(subset=chave).sum())} duplicadas na chave {tuple(chave)}",
        "2. ausencia/codigos":
            f"{int(df.isna().sum().sum())} celulas nulas; SOH>100: {int((df['SOH'] > 100).sum())}",
        "3. tipos/unidades":
            ", ".join(f"{c}={t}" for c, t in df.dtypes.astype(str).items()),
        "4. dominio/plausibilidade":
            f"SOH {df['SOH'].min():.1f}..{df['SOH'].max():.1f} %; "
            f"V {df['average_voltage_V'].min():.2f}..{df['average_voltage_V'].max():.2f}; "
            f"R {df['internal_resistance_ohm'].min():.3f}..{df['internal_resistance_ohm'].max():.3f} ohm",
        "5. cobertura/juncoes":
            f"{df['battery_id'].nunique()} baterias ({', '.join(sorted(df['battery_id'].unique()))}); "
            f"C_rate {sorted(df['C_rate'].unique())}",
    }


calce_raw = pd.read_csv(RAW / "ORIGINAL_nomeErrado_recarregados-emplacamentos.csv")
print("Dimensao CALCE:", calce_raw.shape)
pd.DataFrame(verificar_qualidade(calce_raw, ["battery_id", "cycle_number"]).items(),
             columns=["verificacao", "resultado"])


Dimensao CALCE: (5662, 7)


,verificacao,resultado
0,1. chaves/duplicidade,0 linhas duplicadas; 0 duplicadas na chave ('b...
1,2. ausencia/codigos,0 celulas nulas; SOH>100: 6
2,3. tipos/unidades,"battery_id=str, cycle_number=int64, C_rate=flo..."
3,4. dominio/plausibilidade,SOH 5.1..100.6 %; V 3.30..4.02; R 0.082..0.259...
4,5. cobertura/juncoes,"6 baterias (CS2-33, CS2-34, CS2-35, CS2-36, CS..."


### 2.3 Tratamento e registro das decisoes

Regra de ouro: **nenhum original e alterado**. A limpeza gera copia em `dados/processados/`.

In [8]:
calce = calce_raw.copy()
calce = calce.drop_duplicates()
calce = calce.drop_duplicates(subset=["battery_id", "cycle_number"], keep="first")
calce["SOH_corrigido"] = calce["SOH"].clip(upper=100.0)
calce["soh_acima_100_flag"] = calce["SOH"] > 100
calce = calce.sort_values(["battery_id", "cycle_number"]).reset_index(drop=True)

calce_raw.to_csv(PROC / "calce_cs2_soh.csv", index=False)
calce.to_csv(PROC / "calce_cs2_soh_limpo.csv", index=False)

decisoes = pd.DataFrame([
    {"problema": "Linhas duplicadas", "deteccao": "df.duplicated()",
     "afetados": int(calce_raw.duplicated().sum()), "regra": "nenhuma",
     "consequencia": "sem perda"},
    {"problema": "Duplicidade na chave", "deteccao": "subset=[battery_id, cycle_number]",
     "afetados": int(calce_raw.duplicated(subset=["battery_id", "cycle_number"]).sum()),
     "regra": "nenhuma", "consequencia": "chave unica confirmada"},
    {"problema": "SOH fisicamente > 100%", "deteccao": "SOH > 100",
     "afetados": int((calce_raw["SOH"] > 100).sum()),
     "regra": "manter a linha, limitar a 100 em SOH_corrigido + flag",
     "consequencia": "ciclo preservado, valor sinalizado"},
    {"problema": "Celulas nulas", "deteccao": "isna().sum()",
     "afetados": int(calce_raw.isna().sum().sum()), "regra": "nenhuma",
     "consequencia": "sem imputacao"},
])
print(f"linhas: bruto={len(calce_raw)}  limpo={len(calce)}")
decisoes


linhas: bruto=5662  limpo=5662


,problema,deteccao,afetados,regra,consequencia
0,Linhas duplicadas,df.duplicated(),0,nenhuma,sem perda
1,Duplicidade na chave,"subset=[battery_id, cycle_number]",0,nenhuma,chave unica confirmada
2,SOH fisicamente > 100%,SOH > 100,6,"manter a linha, limitar a 100 em SOH_corrigido...","ciclo preservado, valor sinalizado"
3,Celulas nulas,isna().sum(),0,nenhuma,sem imputacao


### 2.4 Vista de contexto - distribuicao/evolucao do SOH (dado observado)

In [9]:
fig_ctx = go.Figure()
for bid, g in calce.groupby("battery_id"):
    fig_ctx.add_trace(go.Scatter(x=g["cycle_number"], y=g["SOH_corrigido"],
                                 mode="lines", name=bid))
fig_ctx.update_layout(
    title="Contexto: degradacao do SOH por ciclo - CALCE CS2 (dado observado)",
    xaxis_title="Numero do ciclo", yaxis_title="SOH (%)",
    template="plotly_white", height=460, legend_title="Bateria")
fig_ctx.show()


### 2.5 Curva de carga - verificacao de capacidade

Mesmas cinco verificacoes + integracao da corrente no tempo (`int |I| dt`), que sustenta o fator de capacidade.

In [10]:
curva = pd.read_excel(RAW / "ORIGINAL_5.596Ah_charge.xlsx")
curva.columns = [str(c).strip() for c in curva.columns]
t = pd.to_datetime(curva["Date_Time"])
dt_h = t.diff().dt.total_seconds().fillna(0) / 3600.0
cap_ah = float((curva["Current(A)"].abs() * dt_h).sum())

cv = {
    "1. chaves/duplicidade": f"{int(curva['Date_Time'].duplicated().sum())} timestamps repetidos",
    "2. ausencia/codigos": f"{int(curva.isna().sum().sum())} celulas nulas",
    "3. tipos/unidades": ", ".join(f"{c}={d}" for c, d in curva.dtypes.astype(str).items()),
    "4. dominio/plausibilidade": f"V {curva['Voltage(V)'].min():.2f}..{curva['Voltage(V)'].max():.2f}; "
                                 f"I {curva['Current(A)'].min():.2f}..{curva['Current(A)'].max():.2f}; "
                                 f"SOC {curva['SOC'].min():.3f}..{curva['SOC'].max():.3f}",
    "5. cobertura/juncoes": f"{len(curva)} leituras de 1 s; "
                            f"capacidade integrada = {cap_ah:.4f} Ah (nominal 5,596 Ah)",
}
curva.to_csv(PROC / "curva_carga_5p596ah_xlsx.csv", index=False)
print("Capacidade integrada:", round(cap_ah, 4), "Ah  (~ nominal 5,596 Ah)")
pd.DataFrame(cv.items(), columns=["verificacao", "resultado"])


Capacidade integrada: 5.5872 Ah  (~ nominal 5,596 Ah)


,verificacao,resultado
0,1. chaves/duplicidade,1 timestamps repetidos
1,2. ausencia/codigos,0 celulas nulas
2,3. tipos/unidades,"Date_Time=str, Voltage(V)=float64, Current(A)=..."
3,4. dominio/plausibilidade,V 2.93..3.65; I 0.00..6.00; SOC 0.068..1.000
4,5. cobertura/juncoes,4180 leituras de 1 s; capacidade integrada = 5...


In [11]:
fig_curva = make_subplots(specs=[[{"secondary_y": True}]])
fig_curva.add_trace(go.Scatter(x=t, y=curva["Voltage(V)"], name="Tensao (V)"), secondary_y=False)
fig_curva.add_trace(go.Scatter(x=t, y=curva["SOC"], name="SOC (0-1)"), secondary_y=True)
fig_curva.update_yaxes(title_text="Tensao (V)", secondary_y=False)
fig_curva.update_yaxes(title_text="SOC (0-1)", secondary_y=True)
fig_curva.update_xaxes(title_text="Tempo de carga")
fig_curva.update_layout(title="Curva de carga 5,596 Ah - verificacao de capacidade (dado observado)",
                        template="plotly_white", height=440)
fig_curva.show()


## 3. Parametros e cenarios (premissas declaradas)

Nada de parametro "escondido": cada valor tem origem declarada (observado, premissa ou simulado).

In [12]:
FATOR_CAP = 0.80   # SOH de 2a vida -> dado observado (CALCE)
FRACAO_USO = 0.90  # DoD -> premissa
EFICIENCIA = 0.90  # rendimento ida+volta -> premissa

parametros = pd.DataFrame([
    {"parametro": "fator_capacidade (SOH 2a vida)", "valor": FATOR_CAP,
     "unidade": "-", "origem": "CALCE CS2 (observado)"},
    {"parametro": "fracao_uso (DoD)", "valor": FRACAO_USO,
     "unidade": "-", "origem": "premissa declarada"},
    {"parametro": "eficiencia (ida e volta)", "valor": EFICIENCIA,
     "unidade": "-", "origem": "premissa declarada"},
])
parametros


,parametro,valor,unidade,origem
0,fator_capacidade (SOH 2a vida),0.8,-,CALCE CS2 (observado)
1,fracao_uso (DoD),0.9,-,premissa declarada
2,eficiencia (ida e volta),0.9,-,premissa declarada


In [13]:
BANCOS = [
    {"id": "A", "nome": "Banco 2a vida conservador", "Enom": 5.0, "Pmax": 1.0},
    {"id": "B", "nome": "Banco 2a vida expandido", "Enom": 10.0, "Pmax": 3.0},
]

CENARIOS = [
    {"id": "posto_vacinas", "nome": "Sala de vacinas (posto de saude)", "dur": 8.0,
     "cargas": [("Refrigerador de vacinas", 0.20, 1), ("Iluminacao de emergencia", 0.05, 4),
                ("Terminal + impressora", 0.10, 1)]},
    {"id": "telecom_erb", "nome": "Estacao radio-base (telecom)", "dur": 4.0,
     "cargas": [("Transmissor/radio", 0.40, 1), ("Ar-condicionado de emergencia", 0.80, 1)]},
    {"id": "residencia_essencial", "nome": "Residencia - circuito essencial", "dur": 6.0,
     "cargas": [("Geladeira", 0.15, 1), ("Bomba d'agua", 0.50, 1),
                ("Iluminacao", 0.08, 3), ("Roteador/internet", 0.02, 1)]},
    {"id": "sem_carga_critica", "nome": "Caso vazio - sem cargas criticas", "dur": 4.0, "cargas": []},
]

linhas = []
for c in CENARIOS:
    for nome, pot, qtd in c["cargas"]:
        linhas.append({"cenario": c["nome"], "duracao_h": c["dur"], "equipamento": nome,
                       "potencia_kW": pot, "qtd": qtd, "subtotal_kW": round(pot * qtd, 3)})
print("Bancos:", [(b["id"], b["Enom"], b["Pmax"]) for b in BANCOS])
pd.DataFrame(linhas)


Bancos: [('A', 5.0, 1.0), ('B', 10.0, 3.0)]


,cenario,duracao_h,equipamento,potencia_kW,qtd,subtotal_kW
0,Sala de vacinas (posto de saude),8.0,Refrigerador de vacinas,0.20,1,0.20
1,Sala de vacinas (posto de saude),8.0,Iluminacao de emergencia,0.05,4,0.20
2,Sala de vacinas (posto de saude),8.0,Terminal + impressora,0.10,1,0.10
3,Estacao radio-base (telecom),4.0,Transmissor/radio,0.40,1,0.40
4,Estacao radio-base (telecom),4.0,Ar-condicionado de emergencia,0.80,1,0.80
5,Residencia - circuito essencial,6.0,Geladeira,0.15,1,0.15
6,Residencia - circuito essencial,6.0,Bomba d'agua,0.50,1,0.50
7,Residencia - circuito essencial,6.0,Iluminacao,0.08,3,0.24
8,Residencia - circuito essencial,6.0,Roteador/internet,0.02,1,0.02


## 4. Modelo de calculo (indicadores)

```
Pcritica      = soma(potencia_i * quantidade_i)
Edemanda      = Pcritica * duracao
Eutilizavel   = Enom * fator_capacidade * fracao_uso * eficiencia
autonomia     = Eutilizavel / Pcritica          (indefinida se Pcritica = 0)
Enao_atendida = max(0, Edemanda - Eutilizavel)
inviavel por potencia se Pmax < Pcritica
```

In [14]:
def pcritica(cargas):
    return sum(p * q for _, p, q in cargas)


def energia_demanda(pc, dur):
    return pc * dur


def energia_utilizavel(b):
    return b["Enom"] * FATOR_CAP * FRACAO_USO * EFICIENCIA


def autonomia(eu, pc):
    return None if pc == 0 else eu / pc


def nao_atendida(edem, eu):
    return max(0.0, edem - eu)


def avaliar(cen, banco):
    pc = pcritica(cen["cargas"])
    edem = energia_demanda(pc, cen["dur"])
    eu = energia_utilizavel(banco)
    aut = autonomia(eu, pc)
    nao = nao_atendida(edem, eu)
    if pc == 0:
        status = "sem_carga"
    elif banco["Pmax"] < pc:
        status = "inviavel_potencia"
    elif aut is not None and aut >= cen["dur"]:
        status = "atende"
    else:
        status = "nao_atende"
    return {"cenario": cen["nome"], "banco": banco["id"], "Pcritica_kW": round(pc, 3),
            "Edemanda_kWh": round(edem, 3), "Eutilizavel_kWh": round(eu, 3),
            "Pmax_kW": banco["Pmax"], "autonomia_h": None if aut is None else round(aut, 3),
            "Enao_atendida_kWh": round(nao, 3), "status": status}


resultados = pd.DataFrame([avaliar(c, b) for b in BANCOS for c in CENARIOS])
resultados


,cenario,banco,Pcritica_kW,Edemanda_kWh,Eutilizavel_kWh,Pmax_kW,autonomia_h,Enao_atendida_kWh,status
0,Sala de vacinas (posto de saude),A,0.50,4.00,3.24,1.0,6.480,0.76,nao_atende
1,Estacao radio-base (telecom),A,1.20,4.80,3.24,1.0,2.700,1.56,inviavel_potencia
2,Residencia - circuito essencial,A,0.91,5.46,3.24,1.0,3.560,2.22,nao_atende
3,Caso vazio - sem cargas criticas,A,0.00,0.00,3.24,1.0,NaN,0.00,sem_carga
4,Sala de vacinas (posto de saude),B,0.50,4.00,6.48,3.0,12.960,0.00,atende
5,Estacao radio-base (telecom),B,1.20,4.80,6.48,3.0,5.400,0.00,atende
6,Residencia - circuito essencial,B,0.91,5.46,6.48,3.0,7.121,0.00,atende
7,Caso vazio - sem cargas criticas,B,0.00,0.00,6.48,3.0,NaN,0.00,sem_carga


Leitura: o **Banco A** (5 kWh / 1 kW) nao atende aos tres cenarios reais; no de **telecom** falta ainda **potencia** (1,2 kW > 1,0 kW). O **Banco B** (10 kWh / 3 kW) atende a todos.

## 5. Visualizacoes

Cada vista declara pergunta, unidade e premissa. A comparacao principal e a V2; a V3 contextualiza com uma **relacao**.

**V1 - Quais cargas formam a demanda critica?** Barras horizontais por equipamento, com a `Pcritica` anotada. Ordenacao pela contribuicao a demanda.

In [15]:
reais = [c for c in CENARIOS if c["cargas"]]
fig_v1 = make_subplots(rows=1, cols=len(reais), subplot_titles=[c["nome"] for c in reais],
                       horizontal_spacing=0.09)
for i, c in enumerate(reais, 1):
    cargas = list(reversed(c["cargas"]))
    vals = [p * q for _, p, q in cargas]
    fig_v1.add_trace(go.Bar(x=vals, y=[n for n, _, _ in cargas], orientation="h",
                            marker_color="#0B6E4F", text=[f"{v:.2f} kW" for v in vals],
                            textposition="outside", showlegend=False,
                            hovertemplate="%{y}<br>%{x:.2f} kW<extra></extra>"), row=1, col=i)
    xref = "x domain" if i == 1 else f"x{i} domain"
    fig_v1.add_annotation(text=f"Pcritica = {pcritica(c['cargas']):.2f} kW", x=0.5, y=1.16,
                          xref=xref, yref="y domain", showarrow=False, font=dict(size=11))
fig_v1.update_xaxes(title_text="Potencia simultanea (kW)", rangemode="tozero")
fig_v1.update_layout(title="V1. Quais cargas formam a demanda critica?",
                     template="plotly_white", height=560)
fig_v1.show()


**V2 - Qual cenario atende ao horizonte?** Barras de autonomia (A e B) com linha pontilhada da duracao requerida. Barra abaixo da linha = nao atende; cenario inviavel por potencia aparece como tal.

In [16]:
fig_v2 = go.Figure()
for b in BANCOS:
    auts = [avaliar(c, b)["autonomia_h"] for c in reais]
    fig_v2.add_trace(go.Bar(name=f"Banco {b['id']} ({b['Enom']:.0f} kWh)",
                            x=[c["nome"] for c in reais], y=auts,
                            marker_color=("#E1A100" if b["id"] == "A" else "#0B6E4F"),
                            text=[f"{a:.1f} h" for a in auts], textposition="outside"))
fig_v2.add_trace(go.Scatter(name="Duracao requerida", x=[c["nome"] for c in reais],
                            y=[c["dur"] for c in reais], mode="markers+lines",
                            marker=dict(symbol="line-ew", size=24, color="#333333"),
                            line=dict(dash="dot", color="#333333")))
fig_v2.update_yaxes(title_text="Autonomia (h)", rangemode="tozero")
fig_v2.update_layout(title="V2. Qual cenario atende ao horizonte requerido?",
                     template="plotly_white", height=520, legend=dict(orientation="h", y=1.12))
fig_v2.show()


**V3 - Como a autonomia varia com o SOH?** Relacao autonomia x fator de capacidade (cenario telecom). Mostra o SOH minimo em que cada banco alcanca o horizonte.

In [17]:
fatores = np.round(np.linspace(0.6, 1.0, 21), 3)
fig_v3 = go.Figure()
for b in BANCOS:
    eu = b["Enom"] * fatores * FRACAO_USO * EFICIENCIA
    fig_v3.add_trace(go.Scatter(x=fatores, y=eu / pcritica(CENARIOS[1]["cargas"]),
                                mode="lines+markers", name=f"Banco {b['id']}",
                                line=dict(color=("#E1A100" if b["id"] == "A" else "#0B6E4F"), width=3)))
fig_v3.add_hline(y=CENARIOS[1]["dur"], line=dict(dash="dot", color="#333333"),
                 annotation_text="Horizonte = 4 h")
fig_v3.add_vline(x=FATOR_CAP, line=dict(dash="dash", color="#8A8F98"),
                 annotation_text="SOH adotado = 0,80")
fig_v3.update_xaxes(title_text="Fator de capacidade remanescente (SOH)")
fig_v3.update_yaxes(title_text="Autonomia (h)", rangemode="tozero")
fig_v3.update_layout(title="V3. Sensibilidade da autonomia ao SOH (cenario telecom)",
                     template="plotly_white", height=520)
fig_v3.show()


## 6. Casos especiais e dois estados

O professor pede **dois estados recalculados** e o **caso sem parametro/carga**.

In [18]:
# Caso sem carga critica (guarda de denominador zero)
vazio = avaliar(CENARIOS[3], BANCOS[1])
assert avaliar(CENARIOS[3], BANCOS[0])["autonomia_h"] is None
print("Caso vazio ->", vazio["status"], "| autonomia:", vazio["autonomia_h"])

# Dois estados do mesmo painel (banco B): trocar o cenario recalcula tudo
estado_1 = avaliar(CENARIOS[0], BANCOS[1])
estado_2 = avaliar(CENARIOS[1], BANCOS[1])
print("Estado 1 (banco B / posto de vacinas):", estado_1["status"], "|", estado_1["autonomia_h"], "h")
print("Estado 2 (banco B / telecom):        ", estado_2["status"], "|", estado_2["autonomia_h"], "h")
pd.DataFrame([estado_1, estado_2])


Caso vazio -> sem_carga | autonomia: None
Estado 1 (banco B / posto de vacinas): atende | 12.96 h
Estado 2 (banco B / telecom):         atende | 5.4 h


,cenario,banco,Pcritica_kW,Edemanda_kWh,Eutilizavel_kWh,Pmax_kW,autonomia_h,Enao_atendida_kWh,status
0,Sala de vacinas (posto de saude),B,0.5,4.0,6.48,3.0,12.96,0.0,atende
1,Estacao radio-base (telecom),B,1.2,4.8,6.48,3.0,5.40,0.0,atende


## 7. Conferencia independente

1. Reproduz o **exemplo do professor**: 3 kW, 3 h, 6 kWh utilizaveis, Pmax 4 kW -> demanda 9 kWh, autonomia 2 h, nao atendida 3 kWh.
2. Recalcula o cenario **posto de vacinas / banco B** por caminho independente.

In [19]:
falhas = []

# (1) exemplo de referencia do professor
ed = 3.0 * 3.0
au = 6.0 / 3.0
nao = max(0.0, 9.0 - 6.0)
if not (ed == 9.0 and au == 2.0 and nao == 3.0):
    falhas.append("exemplo do professor")

# (2) conta paralela: posto_vacinas / banco B
pc = 0.20 * 1 + 0.05 * 4 + 0.10 * 1
ed2 = pc * 8.0
eu2 = 10.0 * 0.80 * 0.90 * 0.90
au2 = eu2 / pc
r = avaliar(CENARIOS[0], BANCOS[1])
ok = (abs(r["Pcritica_kW"] - pc) < 1e-9 and abs(r["Edemanda_kWh"] - ed2) < 1e-9
      and abs(r["Eutilizavel_kWh"] - eu2) < 1e-9 and abs(r["autonomia_h"] - au2) < 1e-9)
if not ok:
    falhas.append("posto_vacinas / banco B")

print("Exemplo do professor:", ed, "kWh /", au, "h /", nao, "kWh  -> OK")
print("Conta independente :", round(ed2, 2), "kWh /", round(au2, 2), "h  -> bate com o modelo:", ok)
print("\nRESULTADO:", "TODAS AS CONFERENCIAS OK" if not falhas else f"FALHAS: {falhas}")

pd.DataFrame([
    {"caso": "Exemplo do professor", "esperado": "9 kWh / 2 h / 3 kWh nao atendida",
     "obtido": f"{ed} kWh / {au} h / {nao} kWh nao atendida"},
    {"caso": "posto_vacinas banco B (independente)",
     "esperado": f"{ed2:.2f} kWh / {au2:.2f} h",
     "obtido": f"{r['Edemanda_kWh']:.2f} kWh / {r['autonomia_h']:.2f} h"},
])


Exemplo do professor: 9.0 kWh / 2.0 h / 3.0 kWh  -> OK
Conta independente : 4.0 kWh / 12.96 h  -> bate com o modelo: True

RESULTADO: TODAS AS CONFERENCIAS OK


,caso,esperado,obtido
0,Exemplo do professor,9 kWh / 2 h / 3 kWh nao atendida,9.0 kWh / 2.0 h / 3.0 kWh nao atendida
1,posto_vacinas banco B (independente),4.00 kWh / 12.96 h,4.00 kWh / 12.96 h


## 8. Sintese, decisao e limitacoes

- **Achado (cenario):** sob as premissas informadas (SOH 0,80; DoD 0,90; eficiencia 0,90), o **Banco B (10 kWh / 3 kW)** atende aos tres cenarios; o **Banco A (5 kWh / 1 kW)** nao atende a nenhum, e no cenario de telecom falha tambem por **potencia**.
- **Energia x potencia:** o cenario de telecom distingue as duas restricoes - o A teria autonomia 2,70 h (insuficiente), mas `Pmax 1,0 < Pcritica 1,2` ja inviabiliza por potencia.
- **O leitor pode concluir:** qual banco e qual cenario atendem ao horizonte e qual premissa condiciona o resultado.
- **O leitor NAO pode concluir:** economia, seguranca, vida util ou viabilidade comercial; isto e um **exercicio de cenarios**, nao medicao de uma bateria real.

### Como reproduzir
1. Fontes em `dados/raw/` (nunca alteradas);
2. Rodar este notebook do topo (gera `dados/processados/` e HTMLs em `dashboard/vistas/`);
3. Conferencias na secao 7.

### Contribuicoes e declaracao de IA
- _Preencher com nome + tarefa + evidencia de cada integrante._
- _Declarar uso de IA: ferramenta, finalidade, o que foi aproveitado e como foi verificado._


In [20]:
relatorio = {
    "originais_sha256": {f.name: sha256(f) for f in sorted(RAW.iterdir()) if f.is_file()},
    "calce": {
        "linhas_bruto": int(len(calce_raw)), "linhas_limpo": int(len(calce)),
        "duplicatas_removidas": int(calce_raw.duplicated().sum()),
        "SOH_maior_100": int((calce_raw["SOH"] > 100).sum()),
        "nulos": int(calce_raw.isna().sum().sum()),
        "SOH_min": float(calce["SOH_corrigido"].min()),
        "SOH_max": float(calce["SOH_corrigido"].max()),
    },
    "curva_carga": {"linhas": int(len(curva)), "nulos": int(curva.isna().sum().sum()),
                    "capacidade_integrada_Ah": round(cap_ah, 4)},
    "parametros": {"fator_capacidade": FATOR_CAP, "fracao_uso": FRACAO_USO,
                   "eficiencia": EFICIENCIA},
    "bancos": BANCOS,
    "resultados": resultados.to_dict(orient="records"),
}
(BASE / "analise" / "relatorio_qualidade.json").write_text(
    json.dumps(relatorio, indent=2, ensure_ascii=False), encoding="utf-8")

for nome, fig in [("v1_carga_critica", fig_v1), ("v2_autonomia", fig_v2),
                  ("v3_sensibilidade", fig_v3)]:
    fig.write_html(VISTAS / f"{nome}.html", include_plotlyjs="cdn")

print("Relatorio salvo em:", BASE / "analise" / "relatorio_qualidade.json")
print("HTMLs salvos em:", VISTAS)
print("Conclusao: 'nas condicoes simuladas' - nao representa medicao de bateria real.")


Relatorio salvo em: C:\Users\pedro\OneDrive\Desktop\Análise\analise\relatorio_qualidade.json
HTMLs salvos em: C:\Users\pedro\OneDrive\Desktop\Análise\dashboard\vistas
Conclusao: 'nas condicoes simuladas' - nao representa medicao de bateria real.
